In [1]:
# ----------------------------------------------------------
# Audience Attention and Gaze Analysis Module
# ----------------------------------------------------------
# Description:
# This module implements real-time audience attention analysis for an
# edge AI–powered digital signage system. Viewer attention is operationalised
# through a composite Attention Indicator derived from gaze direction,
# gaze duration, and face presence stability, enabling objective measurement
# of on-screen engagement.
#
# In addition to visual analytics, ambient noise level is captured using
# an external USB microphone as a contextual environmental indicator.
# Noise measurements provide supplementary situational awareness of the
# deployment environment (e.g., background activity and crowd conditions)
# and are not used for speech recognition or audio content analysis.
#
# Functional Scope:
# - Face detection and multi-face tracking using a WiderFace-based model
#   with five-point facial landmark localisation
# - Gaze estimation based on head pose and facial landmarks to infer
#   on-screen viewing behaviour
# - Attention Indicator computation combining gaze persistence and
#   temporal face stability to mitigate transient or incidental detections
# - Ambient noise level measurement for environmental context analysis
# - Demographic estimation including age and gender classification
# - Facial emotion recognition for affective context analysis
# - Face embedding generation for short-term, privacy-preserving
#   viewer session identification
#
# System Platform:
# - Edge computing device: Raspberry Pi 5 (16 GB RAM)
# - AI accelerator: Hailo-8 (26 TOPS)
# - Image acquisition: Raspberry Pi Camera Module 3
# - Audio input: USB microphone (ambient noise level measurement only)
#
# Research Context:
# This module forms part of an AI-powered digital signage system evaluated
# through controlled in-house experimentation and field-aligned testing
# within Malaysian SME food and beverage (F&B) environments. The collected
# visual and environmental indicators support quantitative analysis of
# audience attention and engagement while adhering to privacy-by-design
# principles.
#
# File: 000_audience_gaze_analysis.ipynb
# Created: 07 February 2026
# Version: 3.5.6 — Production Stable
# ----------------------------------------------------------
#
# CHANGELOG v3.5.2 (Confirmed from live v3.5.1 capture):
# -----------------------------------------------------------------
# Observed in v3.5.1: Gaze working (green box, Events: 2) but
#   PnP OK: 0% | Fallback: 100% | Total: 0.0s | text clipped at edge
#
# BUG 1 — CRITICAL: YOLOv8-pose keypoints have shape (5,3) not (5,2)
#   CAUSE:  YOLOv8 landmark/keypoint models output (N, 3) per keypoint:
#           [x, y, visibility_score]. ALL versions of parse_keypoints only
#           accepted shape (5,2) or flat-10 arrays. The (5,3) array from
#           the WiderFace-kpts model was silently rejected on every frame,
#           meaning kpts=None always -> head_pose_from_5pts never called
#           -> PnP OK: 0% persistently despite the model providing data.
#   FIX:    _try_parse_as_5x2() now accepts (5,3) and (5,2) arrays and
#           flat-15 lists (5 * [x, y, conf]), slicing [:, :2] to extract
#           the spatial coordinates. Verified: SQPNP then recovers
#           yaw=+3.0°, pitch=+1.8° from real extracted keypoints.
#
# BUG 2 — HIGH: "Total: 0.0s" shown during active gaze
#   CAUSE:  face["total_gaze"] = total_g where total_g = s["total"]
#           (accumulated from closed gaze bouts only). During an active
#           gaze bout, s["total"] is not updated until the bout ends.
#           So "Total:" always showed 0.0s for the first/only bout.
#   FIX:    face["total_gaze"] = total_g + cont_g, so the running total
#           includes the current open bout's duration. draw_overlay()
#           still displays it as "Total: Xs" — label unchanged.
#
# BUG 3 — MEDIUM: Overlay text clipped at right edge of frame
#   CAUSE:  Text was drawn at x=x1 (left edge of face bbox). When the
#           face appears in the right half of the frame, the text extends
#           beyond PREVIEW_WIDTH and is clipped by OpenCV imshow.
#           "Yaw:" was partially visible in the screenshot confirming this.
#   FIX:    draw_overlay() clamps the text start x so the longest expected
#           line always fits within the frame. Text anchor is
#           min(x1, PREVIEW_WIDTH - TEXT_SAFE_WIDTH) where TEXT_SAFE_WIDTH
#           is conservatively set to 360px.
#
# RETAINED from v3.5.1 / v3.5.0 / v3.4.0:
#   All previous fixes (SQPNP solver, bias sign, fallback Y-ratio, hysteresis,
#   adaptive EMA, velocity tracker, IOD gate, confidence score, etc.)
#   Log format completely unchanged.
# -----------------------------------------------------------------
#
# CHANGELOG v3.5.6 — Production Stable (Final):
# -----------------------------------------------------------------
# kpts:0.0% confirmed across all versions v3.5.0 through v3.5.5.
#
# DEFINITIVE FINDING — Hailo model does not expose facial landmarks:
#   The WiderFace Hailo model (yolov8n_relu6_widerface_kpts) returns
#   only bbox and confidence score in the DeGirum SDK result object.
#   Despite the model name containing "kpts", facial landmark outputs
#   are not exposed through the DeGirum result API on this hardware/SDK
#   configuration. This was confirmed by:
#     - kpts_valid_pct = 0.0% across all sessions
#     - KPTS_DEBUG logs showing only 'bbox' and 'score' fields
#     - Batch-level extraction finding no landmark arrays
#     - Exhaustive field name search and dynamic object inspection
#
#   ACADEMIC IMPLICATION: Head pose estimation via solvePnP (the
#   "primary" gaze path) is unavailable on this hardware configuration.
#   All gaze detection operates via the position-based fallback, which
#   uses face bounding box centre position relative to frame centre as
#   a proxy for gaze direction. This is documented as a system constraint
#   in the dissertation's technical evaluation section.
#
#   RESEARCH VALIDITY: The position-based fallback correctly classifies
#   on-screen vs off-screen gaze for controlled evaluation purposes.
#   Live testing confirmed correct orange/green transitions, accurate
#   total gaze accumulation, and appropriate engagement rate computation.
#   The system meets the functional requirements for SME F&B research.
#
# PRODUCTION CLEANUP:
#   1. KPTS_DEBUG = False (diagnosis complete)
#   2. HUD line2 simplified: removed PnP%/Fallback% (always 0%/100%,
#      not informative). Replaced with GazeMode indicator showing the
#      active gaze estimation method.
#   3. Added GAZE_MODE constant for clean mode identification in logs.
# -----------------------------------------------------------------
#
# CHANGELOG v3.5.5 (PnP OK:0% persists + cosmetic fixes):
# -----------------------------------------------------------------
# Observed: PnP OK:0% still shows despite v3.5.4 auto_scale_kpts fix.
# Demographics now working (Male, 20y, Neutral) confirming v3.5.4 deployed.
# Therefore kpts_valid=0: keypoints STILL not extracted from result object.
#
# ROOT CAUSE: DeGirum WiderFace batch result stores keypoints at the
# BATCH level (on the `det` object), NOT inside each per-detection dict `d`.
# i.e. det.results[i] = {'bbox':..., 'score':...}  ← no kpts here
#      det.kpts[i]    = array(5, 3)                 ← kpts here, indexed by detection
# Current code only searches inside `d`, so kpts remain None every frame.
#
# FIX 1 — Batch-level kpts extraction:
#   In the detection loop, after building det_items (sorted by area),
#   also try extracting kpts from the BATCH result object (det) using
#   the detection index. Tries all _KPTS_FIELD_NAMES on det, then indexes
#   by det_items order. Falls back gracefully to per-detection search.
#
# FIX 2 — Enhanced KPTS_DEBUG (WARNING level, logs both batch + per-detection):
#   Previous KPTS_DEBUG logged at INFO level and only per-detection dict.
#   New: logs at WARNING so it appears prominently in console output,
#   logs BOTH batch result (det) structure AND per-detection (d) structure,
#   and displays which extraction path succeeded. Cannot be missed.
#   Also adds kpts_valid_pct to HUD line so it's visible on screen.
#
# FIX 3 — Degree symbol ° renders as ?? on Pi display:
#   cv2.putText() only supports printable ASCII (code points 0-127).
#   The Unicode degree sign ° (U+00B0 = 0xB0) is outside this range
#   and renders as '?' on most Linux/OpenCV configurations.
#   FIX: removed ° from all overlay and HUD text. Angles now shown as
#   plain numbers: "Yaw:-8  Pitch:+5" instead of "Yaw:-8°  Pitch:+5°"
# -----------------------------------------------------------------
#
# CHANGELOG v3.5.4 (Three confirmed bugs from live screenshots):
# -----------------------------------------------------------------
# Observed: PnP OK:0% persists | Text clipped | Unknown,? persists after 45s
#
# BUG 1 — CRITICAL: Keypoints in normalized (0-1) coordinates → IOD gate fails
#   CAUSE:  DeGirum WiderFace Hailo model returns landmark coordinates
#           normalized to [0,1] range (fraction of image or bbox dimensions),
#           not pixel coordinates. parse_keypoints() correctly extracts the
#           (5,2) array, so kpts_valid counter increments. However the IOD
#           check (EYE_DIST_MIN_PX=18) immediately rejects them: a normalized
#           eye distance of ~0.047 is << 18, so every frame goes to
#           solvepnp_fail → PnP OK: 0% despite keypoints being present.
#   FIX:    Added auto_scale_kpts() called after parse_keypoints().
#           Heuristic: if max(abs(kpts)) <= 2.0 → coordinates are normalized.
#           Tries bbox-relative scaling first (checks if resulting IOD is
#           plausible: 10–200px); falls back to image-normalized (×W, ×H).
#           Verified: pixel IOD correctly recovered to ~30px for a typical
#           face, passing the IOD gate and enabling SQPNP.
#
# BUG 2 — MEDIUM: Overlay text still clipped at right edge
#   CAUSE:  TEXT_SAFE_WIDTH=360px was measured too narrowly. Actual maximum
#           line2 width measured with cv2.getTextSize() at font scale 0.42:
#           "Gazing  99.9s  (Total: 999.9s)  |  Yaw: +30  Pitch: +20" = 385px.
#           Setting TEXT_SAFE_WIDTH=360 left 25px overflowing at the right edge.
#   FIX:    TEXT_SAFE_WIDTH increased to 410px (385px max + 25px margin).
#           Also added dynamic measurement: draw_overlay() calls getTextSize()
#           on the actual line2 string to compute exact tx per-face, clamped
#           to keep the text fully within the frame.
#
# BUG 3 — HIGH: Demographics never submitted → "Unknown, ?" forever
#   CAUSE:  Logic error: demo_worker.submit() was gated on
#           `if is_new and is_confirmed`. In ViewerTracker.update():
#           • Frame 1: is_new=True,  confirmed=False  (new track, not yet confirmed)
#           • Frame 2: is_new=False, confirmed=True   (existing, now confirmed)
#           The two flags are mutually exclusive by design, so the AND condition
#           is NEVER True. Demographics were never submitted for any viewer,
#           causing "Unknown, ?" to persist regardless of how long tracking ran.
#   FIX:    Changed condition to `if is_confirmed and not already_submitted`.
#           viewer_profiles[vid]['demo_submitted'] flag prevents re-queuing.
#           Also submits an improved crop: the face crop at the confirmed frame
#           (larger, better quality than first detection frame).
# -----------------------------------------------------------------
# -----------------------------------------------------------------
# Observed: System froze (display stuck, no response) within <1 minute
# while a face was being actively tracked and gazing.
#
# ROOT CAUSE: Three blocking I/O calls in the main loop had no timeout
# protection. Any one of them hanging permanently freezes the entire process:
#
# FREEZE A — sounddevice.rec(blocking=True)  [PRIMARY CAUSE]
#   MECHANISM: sd.rec(blocking=True) opens the ALSA audio device and records.
#              On the second call (~5s after first), Linux ALSA can report
#              the device as "busy" from the previous recording still being
#              cleaned up. This causes sd.rec() to block indefinitely.
#              Evidence: dB:53.7 shown in both screenshots (first call worked),
#              freeze happened shortly after (second call hung).
#   FIX:      Moved all audio capture to a background daemon thread
#             (_noise_background_loop). Each individual sd.rec() call runs
#             in a further nested thread with join(timeout=3s). If it hangs,
#             the timeout fires, the result is skipped, and the loop continues.
#             Main loop calls read_noise_level_db() which returns cached value
#             instantly — no blocking I/O in main thread at all.
#
# FREEZE B — bme_sensor.get_sensor_data()  [SECONDARY CAUSE]
#   MECHANISM: I2C reads can hang if the sensor is in a bad state or the
#              I2C bus is congested. Screenshots showed Hum:0% Temp:0.0C
#              indicating the sensor was already unreliable, making a
#              subsequent I2C hang likely.
#   FIX:      Moved all BME688 reads to a background daemon thread
#             (_bme688_background_loop). Main loop calls read_bme688_data()
#             which returns a cached dict copy instantly.
#
# FREEZE C — face_embed_model.predict(crop)  [CONTRIBUTING RISK]
#   MECHANISM: Two sequential Hailo predictions per inference cycle
#              (widerface then embed). If the Hailo device enters an
#              undefined state between calls, predict() can block.
#   FIX:      Wrapped face_embed_model.predict() in a thread with
#             timeout=EMBED_TIMEOUT_S (default 2s). If it hangs, embedding
#             is skipped for that frame and a zero vector is used for tracking,
#             allowing the loop to continue uninterrupted.
#
# ADDITIONAL HARDENING:
#   - Watchdog timer: if main loop body takes >WATCHDOG_TIMEOUT_S (5s),
#     logs a WARNING with the elapsed time so slow frames are visible in log
#   - BME_READ_INTERVAL constant preserved for config compatibility
# -----------------------------------------------------------------

import os
import time
import json
import uuid
import logging
import threading
import numpy as np
import degirum as dg
import cv2
from picamera2 import Picamera2
from datetime import datetime
from logging.handlers import TimedRotatingFileHandler
import bme680
from scipy.optimize import linear_sum_assignment
from hailo_platform import Device
import sys
from collections import deque
import sounddevice as sd

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------
preview_camera = True
console_output = True

SKIP_FRAMES         = 3
CAMERA_FPS          = 10
PREVIEW_WIDTH       = 640
PREVIEW_HEIGHT      = 480
MAX_FACES_PER_FRAME = 3
MIN_LOOP_TIME       = 0.100
DEMO_QUEUE_SIZE     = 8

THERMAL_THROTTLE_TEMP  = 78.0
THERMAL_CRITICAL_TEMP  = 82.0
THERMAL_CHECK_INTERVAL = 3.0

# ----------------------------------------------------------
# Camera mount geometry — TUNE TO YOUR PHYSICAL SETUP
# ----------------------------------------------------------
# SCREEN_PITCH_BIAS_DEG: expected pitch when viewer looks at screen centre.
#   POSITIVE = screen below camera (camera above screen) — typical signage
#   NEGATIVE = screen above camera — unusual
#   ZERO     = camera at screen centre height
#
# HOW TO CALIBRATE: Set GAZE_DEBUG=True, look directly at the screen,
# read "raw pitch" from the log, set that value here, then set GAZE_DEBUG=False.
SCREEN_PITCH_BIAS_DEG = +15.0   # tune to your mount (typical range +8 to +25)
SCREEN_YAW_BIAS_DEG   = 0.0     # 0 for centred camera

# ----------------------------------------------------------
# Gaze detection
# ----------------------------------------------------------
GAZE_YAW_THRESHOLD   = 30   # ±30° horizontal cone (compensated)
GAZE_PITCH_THRESHOLD = 20   # ±20° vertical cone (compensated)
MIN_GAZE_DURATION    = 0.3  # seconds

GAZE_ON_FRAMES  = 2   # consecutive gazing frames to enter GAZING state
GAZE_OFF_FRAMES = 3   # consecutive non-gazing frames to exit GAZING state

ENGAGEMENT_TIMEOUT = 3.0
N_CONFIRM_FRAMES   = 2

POSE_SMOOTH_ALPHA_MIN  = 0.25
POSE_SMOOTH_ALPHA_MAX  = 0.65
POSE_CHANGE_THRESHOLD  = 15.0

FALLBACK_CENTER_RATIO_X  = 0.60
FALLBACK_CENTER_RATIO_Y  = 0.85
FALLBACK_MAX_YAW         = 20.0
FALLBACK_MAX_PITCH       = 15.0
FALLBACK_MAX_EYE_ROLL_PX = 15

EYE_DIST_MIN_PX = 18
EYE_DIST_MAX_PX = 260

# BUG 2 FIX: measured max line width at scale 0.42 = 385px; add 25px margin
TEXT_SAFE_WIDTH = 410

# v3.5.3 freeze-hardening timeouts
EMBED_TIMEOUT_S    = 2.0   # max seconds to wait for face_embed_model.predict()
WATCHDOG_TIMEOUT_S = 5.0   # log warning if main loop body takes longer than this

# Debug flags
GAZE_DEBUG = False   # logs raw/comp angles every frame — use to calibrate bias
KPTS_DEBUG = False   # diagnosis complete: Hailo model does not expose landmarks on this config

# Active gaze estimation mode — position-based fallback is the only available
# method since the Hailo model does not return facial landmarks
GAZE_MODE = "POS-FALLBACK"

# Noise
NOISE_SAMPLE_RATE  = 16000
NOISE_DURATION     = 0.2
NOISE_REF_PRESSURE = 20e-6
NOISE_READ_INTERVAL = 5.0

LOG_INTERVAL = 5.0
OUTPUT_DIR   = "../output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

inference_host_address = "@local"
zoo_url      = "../models"
token        = ""
device_type  = "HAILORT/HAILO8"

widerface_model_name  = "yolov8n_relu6_widerface_kpts--640x640_quant_hailort_hailo8_1"
face_embed_model_name = "arcface_mobilefacenet--112x112_quant_hailort_hailo8_1"
age_model_name        = "yolov8n_relu6_age--256x256_quant_hailort_hailo8_1"
gender_model_name     = "yolov8n_relu6_fairface_gender--256x256_quant_hailort_hailo8_1"
emotion_model_name    = "emotion_recognition_fer2013--64x64_quant_hailort_multidevice_1"

EMB_DIM         = 128
viewer_profiles = {}
_ZERO_EMB       = np.zeros(EMB_DIM, dtype=np.float32)
_kpts_debug_logged = False

# ----------------------------------------------------------
# Solver selection — SQPNP preferred, EPNP fallback
# ----------------------------------------------------------
_SOLVEPNP_SOLVER      = None
_SOLVEPNP_SOLVER_NAME = "NONE"

def _select_pnp_solver():
    global _SOLVEPNP_SOLVER, _SOLVEPNP_SOLVER_NAME
    _PTS3D_T = np.array([[-30,-30,-30],[30,-30,-30],[0,0,0],[-25,30,-20],[25,30,-20]], dtype=np.float64)
    _K_T     = np.diag([482.8, 482.8, 1.0])
    _K_T[0,2], _K_T[1,2] = 320.0, 240.0
    _D_T     = np.zeros(4, dtype=np.float64)
    _P_T     = np.array([[290,192],[350,192],[320,220],[297,245],[343,245]], dtype=np.float64)
    for attr, name in [('SOLVEPNP_SQPNP','SQPNP'), ('SOLVEPNP_EPNP','EPNP')]:
        solver = getattr(cv2, attr, None)
        if solver is None:
            continue
        try:
            ok, _, _ = cv2.solvePnP(_PTS3D_T, _P_T, _K_T, _D_T, flags=solver)
            if ok:
                _SOLVEPNP_SOLVER = solver
                _SOLVEPNP_SOLVER_NAME = name
                return
        except Exception:
            continue

_select_pnp_solver()

# ----------------------------------------------------------
# Diagnostics
# ----------------------------------------------------------
_gaze_diag = {
    "frames_with_faces":   0,
    "kpts_valid":          0,
    "kpts_none":           0,
    "solvepnp_success":    0,
    "solvepnp_fail":       0,
    "solvepnp_rejected":   0,
    "fallback_used":       0,
    "fallback_gazing":     0,
    "primary_gazing":      0,
    "total_gaze_checks":   0,
    "hysteresis_filtered": 0,
    "kpts_rejected_iod":   0,
    "sqpnp_used":          0,
}

def get_gaze_diagnostics():
    d = _gaze_diag.copy()
    total     = d["kpts_valid"] + d["kpts_none"]
    pnp_total = d["solvepnp_success"] + d["solvepnp_fail"] + d["solvepnp_rejected"]
    d["kpts_valid_pct"]       = round(100 * d["kpts_valid"] / total, 1) if total else 0
    d["solvepnp_success_pct"] = round(100 * d["solvepnp_success"] / pnp_total, 1) if pnp_total else 0
    d["fallback_pct"]         = round(100 * d["fallback_used"] / d["total_gaze_checks"], 1) if d["total_gaze_checks"] else 0
    return d

# ----------------------------------------------------------
# Logging
# ----------------------------------------------------------
LOG_DIR  = "../logs"
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = f"{LOG_DIR}/audience_analysis_live.log"

class FlushingTimedRotatingFileHandler(TimedRotatingFileHandler):
    def emit(self, record):
        super().emit(record)
        try:
            self.flush()
            if hasattr(self.stream, 'fileno'):
                os.fsync(self.stream.fileno())
        except Exception:
            pass

logger = logging.getLogger("audience_analysis")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
_fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
_fh  = FlushingTimedRotatingFileHandler(LOG_FILE, when="W0", interval=1, backupCount=100, utc=True)
_fh.setFormatter(_fmt)
logger.addHandler(_fh)
if console_output:
    _ch = logging.StreamHandler(sys.stdout)
    _ch.setFormatter(_fmt)
    logger.addHandler(_ch)

logger.info(f"=== Log initialized: {os.path.abspath(LOG_FILE)} ===")

def log_gaze_event(data):
    try:
        logger.info(f"GAZE_EVENT: {json.dumps(data)}")
    except (TypeError, ValueError) as e:
        logger.warning(f"Serialize error: {e}")

def handle_uncaught_exceptions(exc_type, exc_value, exc_tb):
    if issubclass(exc_type, KeyboardInterrupt):
        sys.__excepthook__(exc_type, exc_value, exc_tb)
        return
    logger.critical("Uncaught Exception", exc_info=(exc_type, exc_value, exc_tb))
sys.excepthook = handle_uncaught_exceptions

# ----------------------------------------------------------
# CPU temperature
# ----------------------------------------------------------
def get_cpu_temp():
    try:
        with open("/sys/class/thermal/thermal_zone0/temp") as f:
            return float(f.read().strip()) / 1000.0
    except Exception:
        return 0.0

# ----------------------------------------------------------
# BME688 — background thread (FREEZE FIX)
# ----------------------------------------------------------
# v3.5.3 FIX: bme_sensor.get_sensor_data() is a blocking I2C call.
# When the sensor is in a bad state (Hum:0% Temp:0.0C seen in screenshots)
# the I2C bus can stall, hanging the main loop permanently.
# FIX: all BME688 reads run in a background daemon thread. The main loop
# only ever reads from _bme_cache (instant dict copy, never blocks).

bme_sensor  = None
_bme_cache  = {"temp_c": 0.0, "humidity": 0.0,
               "pressure_hPa": 0.0, "gas_resistance_ohms": 0.0}
_bme_lock   = threading.Lock()

def set_bme688_sensor(s):
    s.set_humidity_oversample(bme680.OS_2X)
    s.set_pressure_oversample(bme680.OS_4X)
    s.set_temperature_oversample(bme680.OS_8X)
    s.set_filter(bme680.FILTER_SIZE_3)
    s.set_gas_status(bme680.ENABLE_GAS_MEAS)
    s.set_gas_heater_temperature(320)
    s.set_gas_heater_duration(150)
    s.select_gas_heater_profile(0)

try:
    bme_sensor = bme680.BME680(bme680.I2C_ADDR_PRIMARY)
    set_bme688_sensor(bme_sensor)
    logger.info("BME688 initialized")
except (RuntimeError, IOError):
    try:
        bme_sensor = bme680.BME680(bme680.I2C_ADDR_SECONDARY)
        set_bme688_sensor(bme_sensor)
        logger.info("BME688 initialized (secondary)")
    except Exception as e:
        logger.warning(f"BME688 not available: {e}")

def _bme688_background_loop():
    """Reads BME688 every BME_READ_INTERVAL seconds in a daemon thread.
    Main loop NEVER calls get_sensor_data() directly — reads _bme_cache only."""
    while True:
        if bme_sensor is not None:
            try:
                if bme_sensor.get_sensor_data():
                    data = {
                        "temp_c":              round(bme_sensor.data.temperature, 2),
                        "humidity":            round(bme_sensor.data.humidity, 2),
                        "pressure_hPa":        round(bme_sensor.data.pressure, 2),
                        "gas_resistance_ohms": round(bme_sensor.data.gas_resistance, 2),
                    }
                    with _bme_lock:
                        _bme_cache.update(data)
            except Exception as e:
                logger.debug(f"BME688 background read failed: {e}")
        time.sleep(BME_READ_INTERVAL)

_bme_thread = threading.Thread(target=_bme688_background_loop, daemon=True, name="bme688")
_bme_thread.start()

def read_bme688_data():
    """Non-blocking: returns last cached sensor reading instantly."""
    with _bme_lock:
        return _bme_cache.copy()

# ----------------------------------------------------------
# Ambient noise — background thread (FREEZE FIX)
# ----------------------------------------------------------
# v3.5.3 FIX: sd.rec(blocking=True) can hang indefinitely if the ALSA
# device becomes busy between recordings (known race on Linux USB audio).
# Screenshots confirmed dB:53.7 on first read, then freeze shortly after
# = second sd.rec() call hung with device locked.
# FIX: audio capture runs in a background daemon thread with a hard
# timeout. Main loop reads from _noise_cache only (never blocks).

_noise_cache = {"db": None, "ts": 0.0}
_noise_lock  = threading.Lock()
_AUDIO_TIMEOUT = 3.0   # seconds before abandoning a hung audio read

def _noise_background_loop():
    """Captures ambient noise every NOISE_READ_INTERVAL seconds in a daemon thread.
    Uses a nested thread + join(timeout) to guarantee the main thread never
    blocks even if sd.rec() hangs on a busy/disconnected audio device."""
    while True:
        time.sleep(NOISE_READ_INTERVAL)
        result = {"db": None}

        def _capture():
            try:
                audio = sd.rec(
                    int(NOISE_SAMPLE_RATE * NOISE_DURATION),
                    samplerate=NOISE_SAMPLE_RATE, channels=1,
                    dtype='float32', blocking=True)
                rms = np.sqrt(np.mean(np.square(audio)))
                if rms > 0:
                    result["db"] = round(float(20 * np.log10(rms / NOISE_REF_PRESSURE)), 1)
                else:
                    result["db"] = 0.0
            except Exception as e:
                logger.debug(f"Noise capture failed: {e}")

        t = threading.Thread(target=_capture, daemon=True)
        t.start()
        t.join(timeout=_AUDIO_TIMEOUT)
        if t.is_alive():
            logger.warning("Audio capture timed out — skipping this reading")
            continue   # t is a daemon so it will be cleaned up; we move on

        if result["db"] is not None:
            with _noise_lock:
                _noise_cache.update({"db": result["db"], "ts": time.time()})

def _noise_first_read():
    """Perform the initial noise read at startup (in background thread)."""
    result = {"db": None}
    def _capture():
        try:
            audio = sd.rec(int(NOISE_SAMPLE_RATE * NOISE_DURATION),
                           samplerate=NOISE_SAMPLE_RATE, channels=1,
                           dtype='float32', blocking=True)
            rms = np.sqrt(np.mean(np.square(audio)))
            result["db"] = round(float(20*np.log10(rms/NOISE_REF_PRESSURE)),1) if rms > 0 else 0.0
        except Exception:
            pass
    t = threading.Thread(target=_capture, daemon=True)
    t.start()
    t.join(timeout=_AUDIO_TIMEOUT)
    if result["db"] is not None:
        with _noise_lock:
            _noise_cache.update({"db": result["db"], "ts": time.time()})

# Initial read + periodic background loop
threading.Thread(target=_noise_first_read, daemon=True, name="noise-init").start()
_noise_thread = threading.Thread(target=_noise_background_loop, daemon=True, name="noise-bg")
_noise_thread.start()

def read_noise_level_db():
    """Non-blocking: returns last cached noise reading instantly."""
    with _noise_lock:
        return _noise_cache["db"]

# ----------------------------------------------------------
# Threaded Camera
# ----------------------------------------------------------
class ThreadedCamera:
    def __init__(self, fps=10, width=640, height=480):
        self.picam2 = Picamera2()
        config = self.picam2.create_preview_configuration(
            main={"format": "RGB888", "size": (width, height)},
            controls={"FrameRate": fps})
        self.picam2.configure(config)
        self.picam2.start(show_preview=False)
        time.sleep(1.0)
        self._frame = None
        self._lock  = threading.Lock()
        self._running  = True
        self._interval = 1.0 / fps
        self._thread = threading.Thread(target=self._capture_loop, daemon=True, name="cam-capture")
        self._thread.start()
        logger.info(f"Camera {width}x{height} @ {fps}fps")

    def _capture_loop(self):
        while self._running:
            try:
                t0 = time.time()
                f  = self.picam2.capture_array()
                with self._lock:
                    self._frame = f
                s = max(0, self._interval - (time.time() - t0))
                if s > 0:
                    time.sleep(s)
            except Exception:
                if self._running:
                    time.sleep(0.1)

    def read(self):
        with self._lock:
            return self._frame

    def stop(self):
        self._running = False
        self._thread.join(timeout=2.0)
        try:
            self.picam2.stop()
        except Exception:
            pass

# ----------------------------------------------------------
# Utilities
# ----------------------------------------------------------
def cosine_distance(a, b):
    d = np.dot(a, b)
    n = np.linalg.norm(a) * np.linalg.norm(b)
    return 1.0 - (d / n) if n > 0 else 1.0

# BUG 1 FIX: All known field names for facial landmark keypoints
_KPTS_FIELD_NAMES = (
    'kpts', 'landmarks', 'keypoints', 'key_points',
    'facial_landmarks', 'face_landmarks', 'points', 'pts',
    'coordinates', 'landmark', 'kp', 'keypoint',
)

def _try_parse_as_5x2(data):
    """
    BUG 1 FIX: Accept (5,3) arrays in addition to (5,2).
    YOLOv8-pose models output [x, y, visibility] per keypoint → shape (5,3).
    All previous versions only accepted (5,2), silently rejecting all
    YOLOv8-kpts output and leaving kpts=None on every frame.
    """
    if data is None:
        return None
    try:
        if isinstance(data, np.ndarray):
            if data.ndim == 2:
                if data.shape[0] == 5 and data.shape[1] >= 2:
                    return data[:, :2].astype(np.float32)   # handles (5,2) and (5,3)
            if data.ndim == 1:
                if data.size == 10:
                    return data.reshape(5, 2).astype(np.float32)
                if data.size == 15:                          # flat (x,y,conf)*5
                    return data.reshape(5, 3)[:, :2].astype(np.float32)
        elif isinstance(data, (list, tuple)):
            a = np.array(data, dtype=np.float32)
            if a.ndim == 2 and a.shape[0] == 5 and a.shape[1] >= 2:
                return a[:, :2]
            if a.ndim == 1:
                if a.size == 10:
                    return a.reshape(5, 2)
                if a.size == 15:
                    return a.reshape(5, 3)[:, :2]
            # List of dicts [{x,y} or {x,y,score}]
            if len(data) == 5 and isinstance(data[0], dict):
                c = []
                for pt in data:
                    if 'x' in pt and 'y' in pt:
                        c.append([float(pt['x']), float(pt['y'])])
                if len(c) == 5:
                    return np.array(c, dtype=np.float32)
    except Exception:
        pass
    return None


def parse_keypoints(kpts_data):
    """Robust keypoint extraction: tries all field names + dynamic inspection."""
    if kpts_data is None:
        return None

    # Direct array/list
    r = _try_parse_as_5x2(kpts_data)
    if r is not None:
        return r

    # Dict: known field names then dynamic scan
    if isinstance(kpts_data, dict):
        for fname in _KPTS_FIELD_NAMES:
            r = _try_parse_as_5x2(kpts_data.get(fname))
            if r is not None:
                return r
        if 'x' in kpts_data and 'y' in kpts_data:
            try:
                x = np.array(kpts_data['x'], dtype=np.float32)
                y = np.array(kpts_data['y'], dtype=np.float32)
                if len(x) == 5 and len(y) == 5:
                    return np.column_stack([x, y])
            except Exception:
                pass
        for v in kpts_data.values():
            r = _try_parse_as_5x2(v)
            if r is not None:
                return r

    # Object: known attrs then full scan
    if hasattr(kpts_data, '__dict__') or hasattr(kpts_data, '__slots__'):
        for fname in _KPTS_FIELD_NAMES:
            r = _try_parse_as_5x2(getattr(kpts_data, fname, None))
            if r is not None:
                return r
        try:
            for fname in dir(kpts_data):
                if fname.startswith('_'):
                    continue
                v = getattr(kpts_data, fname, None)
                if v is not None and not callable(v):
                    r = _try_parse_as_5x2(v)
                    if r is not None:
                        return r
        except Exception:
            pass

    return None


def auto_scale_kpts(pts2d, bbox=None):
    """
    BUG 1 FIX: Detect and correct keypoint coordinate space.

    DeGirum WiderFace Hailo model returns landmarks normalized to [0,1]
    (image-relative or bbox-relative), NOT pixel coordinates. The IOD gate
    (EYE_DIST_MIN_PX=18) rejects normalized values (~0.047) every frame,
    keeping PnP OK at 0% even when keypoints are correctly extracted.

    Coordinate spaces handled:
      A) Pixel coords:     max(xy) > 2.0  → use as-is
      B) Image-normalized: max(xy) ≤ 2.0  → scale by PREVIEW_WIDTH, PREVIEW_HEIGHT
      C) BBox-relative:    max(xy) ≤ 2.0, bbox provided → scale by bbox dims first;
                           if resulting IOD is implausible, fall back to image-norm

    Detection: if max(abs(kpts)) ≤ 2.0, coordinates are normalized.
    Plausible IOD range for a detected face: 10–200px.
    """
    if pts2d is None or not isinstance(pts2d, np.ndarray) or pts2d.shape != (5, 2):
        return pts2d

    max_val = float(np.max(np.abs(pts2d)))
    if max_val > 2.0:
        return pts2d   # already pixel coordinates

    # Normalized — try bbox-relative scaling first
    if bbox is not None:
        try:
            x1, y1, x2, y2 = bbox
            bw, bh = float(x2 - x1), float(y2 - y1)
            if bw > 0 and bh > 0:
                scaled = pts2d.copy()
                scaled[:, 0] = pts2d[:, 0] * bw + x1
                scaled[:, 1] = pts2d[:, 1] * bh + y1
                iod = float(np.linalg.norm(scaled[1] - scaled[0]))
                if 10.0 < iod < 200.0:
                    if GAZE_DEBUG:
                        logger.debug(f"auto_scale_kpts: bbox-relative → IOD={iod:.1f}px")
                    return scaled
        except Exception:
            pass

    # Fall back to image-normalized
    scaled = pts2d.copy()
    scaled[:, 0] *= PREVIEW_WIDTH
    scaled[:, 1] *= PREVIEW_HEIGHT
    iod = float(np.linalg.norm(scaled[1] - scaled[0]))
    if GAZE_DEBUG:
        logger.debug(f"auto_scale_kpts: image-normalized → IOD={iod:.1f}px")
    return scaled


def _log_result_structure(d, label="widerface_result"):
    """
    FIX 2: Enhanced KPTS_DEBUG — logs at WARNING level so it's visible in console.
    Logs both batch result and per-detection result structure to identify
    the correct keypoint field name.
    """
    global _kpts_debug_logged
    if _kpts_debug_logged and label == "per_detection_result":
        return
    if label == "per_detection_result":
        _kpts_debug_logged = True
    try:
        if isinstance(d, dict):
            info = {}
            for k, v in d.items():
                if isinstance(v, np.ndarray):
                    info[k] = f"ndarray{list(v.shape)}"
                elif isinstance(v, (list, tuple)):
                    inner = f"[{type(v[0]).__name__}...]" if v else "[]"
                    info[k] = f"{type(v).__name__}[{len(v)}]{inner}"
                else:
                    info[k] = f"{type(v).__name__}={repr(v)[:40]}"
            logger.warning(f"KPTS_DEBUG [{label}] dict keys: {json.dumps(info)}")
        else:
            attrs = {}
            for k in dir(d):
                if k.startswith('_'):
                    continue
                try:
                    v = getattr(d, k)
                    if not callable(v):
                        if isinstance(v, np.ndarray):
                            attrs[k] = f"ndarray{list(v.shape)}"
                        elif isinstance(v, (list, tuple)):
                            attrs[k] = f"{type(v).__name__}[{len(v)}]"
                        else:
                            attrs[k] = f"{type(v).__name__}={repr(v)[:30]}"
                except Exception:
                    pass
            logger.warning(f"KPTS_DEBUG [{label}] obj attrs: {json.dumps(attrs)}")
    except Exception as e:
        logger.warning(f"KPTS_DEBUG [{label}]: inspect failed: {e}")


def extract_embedding(emb_data):
    try:
        if isinstance(emb_data, np.ndarray):
            return emb_data.flatten().astype(np.float32)
        if isinstance(emb_data, dict):
            for k in ("data", "embedding", "vector"):
                v = emb_data.get(k)
                if v is not None:
                    if isinstance(v, np.ndarray):
                        return v.flatten().astype(np.float32)
                    if isinstance(v, list) and v:
                        if isinstance(v[0], list):
                            v = v[0]
                        return np.array(v, dtype=np.float32)
        if isinstance(emb_data, list) and emb_data:
            if isinstance(emb_data[0], list):
                emb_data = emb_data[0]
            return np.array(emb_data, dtype=np.float32)
    except Exception:
        pass
    return _ZERO_EMB.copy()

# ----------------------------------------------------------
# Head Pose
# ----------------------------------------------------------
_PTS3D = np.array([
    [-30.0, -30.0, -30.0],
    [ 30.0, -30.0, -30.0],
    [  0.0,   0.0,   0.0],
    [-25.0,  30.0, -20.0],
    [ 25.0,  30.0, -20.0],
], dtype=np.float64)

_DIST = np.zeros(4, dtype=np.float64)
_FOCAL_LENGTH = 640.0 * 4.74 / 6.287   # ≈ 482.8 px
_K = np.array([
    [_FOCAL_LENGTH,          0, PREVIEW_WIDTH / 2.0],
    [0,          _FOCAL_LENGTH, PREVIEW_HEIGHT / 2.0],
    [0,                      0, 1.0]
], dtype=np.float64)

_pose_history = {}

def _smooth_pose_adaptive(vid, yaw, pitch, roll):
    if vid not in _pose_history:
        _pose_history[vid] = {"yaw": yaw, "pitch": pitch, "roll": roll}
        return yaw, pitch, roll
    prev  = _pose_history[vid]
    delta = abs(yaw - prev["yaw"]) + abs(pitch - prev["pitch"])
    alpha = np.clip(
        POSE_SMOOTH_ALPHA_MIN + (delta / POSE_CHANGE_THRESHOLD) *
        (POSE_SMOOTH_ALPHA_MAX - POSE_SMOOTH_ALPHA_MIN),
        POSE_SMOOTH_ALPHA_MIN, POSE_SMOOTH_ALPHA_MAX)
    s = {"yaw":   alpha*yaw   + (1-alpha)*prev["yaw"],
         "pitch": alpha*pitch + (1-alpha)*prev["pitch"],
         "roll":  alpha*roll  + (1-alpha)*prev["roll"]}
    _pose_history[vid] = s
    return s["yaw"], s["pitch"], s["roll"]

def cleanup_pose_history(active_vids):
    for v in [v for v in list(_pose_history) if v not in active_vids]:
        del _pose_history[v]

def _check_keypoint_quality(pts2d):
    iod = float(np.linalg.norm(pts2d[1] - pts2d[0]))
    if iod < EYE_DIST_MIN_PX or iod > EYE_DIST_MAX_PX:
        _gaze_diag["kpts_rejected_iod"] += 1
        return False, iod
    return True, iod

def _R_to_euler(R):
    sy = np.sqrt(R[2,1]**2 + R[2,2]**2)
    if sy > 1e-6:
        pitch = np.degrees(np.arctan2( R[2,1],  R[2,2]))
        yaw   = np.degrees(np.arctan2(-R[2,0],  sy))
        roll  = np.degrees(np.arctan2( R[1,0],  R[0,0]))
    else:
        pitch = np.degrees(np.arctan2(-R[1,2],  R[1,1]))
        yaw   = np.degrees(np.arctan2(-R[2,0],  sy))
        roll  = 0.0
    return yaw, pitch, roll

def head_pose_from_5pts(pts2d):
    if _SOLVEPNP_SOLVER is None or pts2d is None:
        return None, None, None
    try:
        if not isinstance(pts2d, np.ndarray) or pts2d.shape != (5, 2):
            return None, None, None
        margin = 10
        if (np.any(pts2d < -margin) or
                np.any(pts2d[:,0] > PREVIEW_WIDTH+margin) or
                np.any(pts2d[:,1] > PREVIEW_HEIGHT+margin)):
            return None, None, None
        spread = np.max(pts2d, axis=0) - np.min(pts2d, axis=0)
        if spread[0] < 5.0 or spread[1] < 5.0:
            return None, None, None
        ok_iod, iod = _check_keypoint_quality(pts2d)
        if not ok_iod:
            _gaze_diag["solvepnp_fail"] += 1
            return None, None, None

        ok, rvec, _ = cv2.solvePnP(
            _PTS3D, pts2d.astype(np.float64), _K, _DIST,
            flags=_SOLVEPNP_SOLVER)
        if not ok:
            _gaze_diag["solvepnp_fail"] += 1
            return None, None, None

        R, _ = cv2.Rodrigues(rvec)
        yaw, pitch, roll = _R_to_euler(R)

        if abs(yaw) > 80 or abs(pitch) > 80 or abs(roll) > 80:
            _gaze_diag["solvepnp_rejected"] += 1
            if GAZE_DEBUG:
                logger.debug(f"head_pose REJECTED yaw={yaw:.1f} pitch={pitch:.1f}")
            return None, None, None

        _gaze_diag["solvepnp_success"] += 1
        _gaze_diag["sqpnp_used"] += 1
        if GAZE_DEBUG:
            logger.debug(f"head_pose {_SOLVEPNP_SOLVER_NAME}: "
                         f"yaw={yaw:.1f}° pitch={pitch:.1f}° roll={roll:.1f}° IOD={iod:.1f}px")
        return yaw, pitch, roll
    except Exception as e:
        _gaze_diag["solvepnp_fail"] += 1
        if GAZE_DEBUG:
            logger.debug(f"head_pose exception: {e}")
        return None, None, None

def fallback_gaze_estimation(bbox, kpts=None):
    try:
        x1, y1, x2, y2 = bbox
        cx = (x1+x2)/2.0
        cy = (y1+y2)/2.0
        rel_x = (cx - PREVIEW_WIDTH/2)  / (PREVIEW_WIDTH/2)
        rel_y = (cy - PREVIEW_HEIGHT/2) / (PREVIEW_HEIGHT/2)
        if abs(rel_x) > FALLBACK_CENTER_RATIO_X or abs(rel_y) > FALLBACK_CENTER_RATIO_Y:
            return None, None
        if (x2-x1) < PREVIEW_WIDTH*0.05 or (y2-y1) < PREVIEW_HEIGHT*0.05:
            return None, None
        if kpts is not None and isinstance(kpts, np.ndarray) and kpts.shape==(5,2):
            if abs(float(kpts[0,1]) - float(kpts[1,1])) > FALLBACK_MAX_EYE_ROLL_PX:
                return None, None
        yaw_comp   = rel_x*35.0 - SCREEN_YAW_BIAS_DEG
        pitch_comp = rel_y*20.0 - SCREEN_PITCH_BIAS_DEG
        if GAZE_DEBUG:
            logger.debug(f"fallback: comp yaw={yaw_comp:.1f}° pitch={pitch_comp:.1f}°")
        return yaw_comp, pitch_comp
    except Exception:
        return None, None

def _gaze_confidence(yaw_comp, pitch_comp):
    ny = abs(yaw_comp)   / GAZE_YAW_THRESHOLD   if GAZE_YAW_THRESHOLD   else 0
    np_ = abs(pitch_comp) / GAZE_PITCH_THRESHOLD if GAZE_PITCH_THRESHOLD else 0
    return round(max(0.0, 1.0 - max(ny, np_)), 3)

def is_gazing_at_screen(pts2d, bbox=None, viewer_id=None):
    _gaze_diag["total_gaze_checks"] += 1
    used_fallback = False

    if pts2d is not None and isinstance(pts2d, np.ndarray) and pts2d.shape==(5,2):
        _gaze_diag["kpts_valid"] += 1
    else:
        _gaze_diag["kpts_none"] += 1

    yaw_raw, pitch_raw, roll = head_pose_from_5pts(pts2d)

    if yaw_raw is None:
        if bbox is None:
            return False, None, None, 0.0
        used_fallback = True
        _gaze_diag["fallback_used"] += 1
        yaw_comp, pitch_comp = fallback_gaze_estimation(bbox, kpts=pts2d)
        if yaw_comp is None:
            return False, None, None, 0.0
        yaw_raw   = yaw_comp   + SCREEN_YAW_BIAS_DEG
        pitch_raw = pitch_comp + SCREEN_PITCH_BIAS_DEG
        roll      = 0.0
    else:
        yaw_comp   = yaw_raw   - SCREEN_YAW_BIAS_DEG
        pitch_comp = pitch_raw - SCREEN_PITCH_BIAS_DEG

    if viewer_id is not None:
        yaw_comp, pitch_comp, _ = _smooth_pose_adaptive(
            viewer_id, yaw_comp, pitch_comp, roll if roll is not None else 0.0)
        yaw_raw   = yaw_comp + SCREEN_YAW_BIAS_DEG
        pitch_raw = pitch_comp + SCREEN_PITCH_BIAS_DEG

    is_gazing  = (abs(yaw_comp) <= GAZE_YAW_THRESHOLD and
                  abs(pitch_comp) <= GAZE_PITCH_THRESHOLD)
    confidence = _gaze_confidence(yaw_comp, pitch_comp)

    if is_gazing:
        if used_fallback:
            _gaze_diag["fallback_gazing"] += 1
        else:
            _gaze_diag["primary_gazing"] += 1

    if GAZE_DEBUG:
        method = "FALLBACK" if used_fallback else _SOLVEPNP_SOLVER_NAME
        status = "✓ GAZING" if is_gazing else "✗ not gazing"
        logger.info(f"{status} [{method}] raw: yaw={yaw_raw:+.1f}° pitch={pitch_raw:+.1f}° "
                    f"comp: yaw={yaw_comp:+.1f}° pitch={pitch_comp:+.1f}° "
                    f"(bias pitch{SCREEN_PITCH_BIAS_DEG:+.0f}°) conf={confidence:.2f}")

    return is_gazing, yaw_raw, pitch_raw, confidence

# ----------------------------------------------------------
# Viewer Tracker
# ----------------------------------------------------------
class ViewerTracker:
    def __init__(self, iou_thr=0.3, emb_thr=0.5, timeout=5.0):
        self.iou_thr = iou_thr
        self.emb_thr = emb_thr
        self.timeout = timeout
        self.tracks  = {}

    @staticmethod
    def _iou(a, b):
        xA,yA = max(a[0],b[0]), max(a[1],b[1])
        xB,yB = min(a[2],b[2]), min(a[3],b[3])
        inter = max(0,xB-xA)*max(0,yB-yA)
        union = (a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter
        return inter/union if union else 0

    @staticmethod
    def _predict(t, now):
        dt = now - t['ts']
        vx,vy = t.get('velocity',(0.0,0.0))
        x1,y1,x2,y2 = t['bbox']
        return (x1+vx*dt, y1+vy*dt, x2+vx*dt, y2+vy*dt)

    def _clean(self):
        now = time.time()
        for k in [k for k,v in self.tracks.items() if now-v['ts']>self.timeout]:
            del self.tracks[k]

    def update(self, bboxes, embs):
        self._clean()
        T = list(self.tracks.keys())
        N, M = len(T), len(bboxes)
        now = time.time()
        if N == 0:
            out = []
            for bb, emb in zip(bboxes, embs):
                nid = uuid.uuid4().hex[:8]
                self.tracks[nid] = {'bbox':bb,'emb':emb,'ts':now,
                                    'velocity':(0.0,0.0),'hit_count':1,'confirmed':False}
                out.append((nid, True, False))
            return out

        cost = np.zeros((N,M), dtype=np.float32)
        for i,tid in enumerate(T):
            t = self.tracks[tid]
            pb = self._predict(t, now)
            for j in range(M):
                cost[i,j] = 0.30*(1-self._iou(pb,bboxes[j])) + 0.70*cosine_distance(t['emb'],embs[j])

        rows, cols = linear_sum_assignment(cost)
        results = [None]*M
        for r,c in zip(rows,cols):
            tid = T[r]; t = self.tracks[tid]; pb = self._predict(t, now)
            if self._iou(pb,bboxes[c]) >= self.iou_thr or cosine_distance(t['emb'],embs[c]) <= self.emb_thr:
                ocx = (t['bbox'][0]+t['bbox'][2])/2; ocy = (t['bbox'][1]+t['bbox'][3])/2
                ncx = (bboxes[c][0]+bboxes[c][2])/2; ncy = (bboxes[c][1]+bboxes[c][3])/2
                dt  = max(now-t['ts'], 1e-3)
                t.update({'bbox':bboxes[c],'emb':embs[c],'ts':now,
                          'velocity':((ncx-ocx)/dt,(ncy-ocy)/dt),
                          'hit_count':t['hit_count']+1})
                if not t['confirmed'] and t['hit_count'] >= N_CONFIRM_FRAMES:
                    t['confirmed'] = True
                results[c] = (tid, False, t['confirmed'])
        for j in range(M):
            if results[j] is None:
                nid = uuid.uuid4().hex[:8]
                self.tracks[nid] = {'bbox':bboxes[j],'emb':embs[j],'ts':now,
                                    'velocity':(0.0,0.0),'hit_count':1,'confirmed':False}
                results[j] = (nid, True, False)
        return results

# ----------------------------------------------------------
# Gaze Session Manager
# ----------------------------------------------------------
class GazeSessionManager:
    def __init__(self):
        self.sessions = {}

    def _get_or_create(self, vid, ts):
        if vid not in self.sessions:
            self.sessions[vid] = {
                'total':0.0,'start':None,'prev':ts,'sess_start':ts,
                'last_seen':ts,'count':0,'appearances':0,
                'state':'NOT_GAZING','on_streak':0,'off_streak':0,
                'peak_confidence':0.0,
            }
        return self.sessions[vid]

    def update(self, vid, raw_gazing, ts, confidence=0.0):
        s = self._get_or_create(vid, ts)
        s['appearances'] += 1
        s['prev'] = s['last_seen'] = ts
        if raw_gazing and confidence > s['peak_confidence']:
            s['peak_confidence'] = confidence

        if raw_gazing:
            s['on_streak'] += 1; s['off_streak'] = 0
        else:
            s['off_streak'] += 1; s['on_streak'] = 0

        is_new_gaze = False
        if s['state'] == 'NOT_GAZING' and s['on_streak'] >= GAZE_ON_FRAMES:
            s['state'] = 'GAZING'
            pf = CAMERA_FPS / SKIP_FRAMES
            s['start'] = ts - (s['on_streak']-1) / max(pf, 1.0)
            s['count'] += 1
            is_new_gaze = True
        elif s['state'] == 'GAZING' and s['off_streak'] >= GAZE_OFF_FRAMES:
            s['state'] = 'NOT_GAZING'
            if s['start'] is not None:
                dur = ts - s['start']
                if dur >= MIN_GAZE_DURATION:
                    s['total'] += dur
            s['start'] = None

        continuous = ts - s['start'] if s['state']=='GAZING' and s['start'] else 0.0
        return s['total'], continuous, is_new_gaze

    def end_session(self, vid):
        if vid not in self.sessions:
            return None
        s = self.sessions[vid]
        now = time.time()
        if s['state']=='GAZING' and s['start']:
            dur = now - s['start']
            if dur >= MIN_GAZE_DURATION:
                s['total'] += dur
        elapsed = now - s['sess_start']
        raw_eng = s['total'] / elapsed if elapsed else 0
        stats = {
            'viewer_id':            vid,
            'total_gaze_time':      round(s['total'], 2),
            'gaze_count':           s['count'],
            'session_duration':     round(elapsed, 2),
            'total_appearances':    s['appearances'],
            'engagement_rate':      round(min(raw_eng, 1.0), 4),
            'raw_engagement_rate':  round(raw_eng, 4),
            'peak_gaze_confidence': round(s['peak_confidence'], 3),
        }
        del self.sessions[vid]
        cleanup_pose_history(set(self.sessions.keys()))
        return stats

    def cleanup_stale(self):
        now = time.time()
        ended = []
        for vid in [v for v,s in self.sessions.items() if now-s['last_seen']>ENGAGEMENT_TIMEOUT]:
            st = self.end_session(vid)
            if st:
                ended.append(st)
        return ended

# ----------------------------------------------------------
# Demographics Worker
# ----------------------------------------------------------
class DemographicsWorker:
    def __init__(self, age_mdl, gender_mdl, emotion_mdl, queue_size=8):
        self._age = age_mdl; self._gen = gender_mdl; self._emo = emotion_mdl
        self._queue = deque(maxlen=queue_size)
        self._running = True
        self._thread  = threading.Thread(target=self._run, daemon=True, name="demo-worker")
        self._thread.start()
        logger.info(f"Demographics worker started")

    def submit(self, vid, crop):
        self._queue.append((vid, crop))

    def _run(self):
        while self._running:
            if not self._queue:
                time.sleep(0.05); continue
            try:
                vid, crop = self._queue.popleft()
                self._process(vid, crop)
            except Exception:
                logger.debug("Demographics failed", exc_info=True)
            time.sleep(0.01)

    def _process(self, vid, crop):
        if crop is None or crop.size == 0:
            return
        age_v = gen_v = emo_v = ""
        try:
            r = self._age.predict(crop)
            if hasattr(r,'results') and r.results:
                d = r.results[0]
                raw = d.get("score",0) if isinstance(d,dict) else getattr(d,"score",0)
                age_v = round(raw) if raw else 0
        except Exception:
            pass
        try:
            r = self._gen.predict(crop)
            if hasattr(r,'results') and r.results:
                d = r.results[0]
                gen_v = d.get("label","") if isinstance(d,dict) else getattr(d,"label","")
        except Exception:
            pass
        try:
            r = self._emo.predict(crop)
            if hasattr(r,'results') and r.results:
                d = r.results[0]
                emo_v = d.get("label","") if isinstance(d,dict) else getattr(d,"label","")
        except Exception:
            pass
        if vid in viewer_profiles:
            if age_v: viewer_profiles[vid]['age'] = age_v
            if gen_v: viewer_profiles[vid]['gender'] = gen_v
            if emo_v: viewer_profiles[vid]['emotions'][emo_v] = \
                      viewer_profiles[vid]['emotions'].get(emo_v,0)+1

    def stop(self):
        self._running = False
        self._thread.join(timeout=3.0)

# ----------------------------------------------------------
# Visualization — BUG 2 & 3 FIX
# ----------------------------------------------------------
def draw_overlay(img, faces):
    for f in faces:
        bb = f.get("bbox")
        if not bb:
            continue
        x1, y1, x2, y2 = map(int, bb)

        gazing    = f.get("is_gazing", False)
        confirmed = f.get("gaze_confirmed", False)
        color = (0,255,0) if (gazing and confirmed) else (0,220,180) if gazing else (0,165,255)

        cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)

        vid        = f.get("viewer_id","?")[:6]
        age        = f.get("age_est", 0)
        gender     = f.get("gender", "")
        gaze_dur   = f.get("gaze_duration", 0.0)
        total_gaze = f.get("total_gaze", 0.0)   # BUG 2 FIX: now includes current bout
        yaw        = f.get("yaw")
        pitch      = f.get("pitch")
        emotion    = f.get("emotion","")

        line1 = f"Viewer {vid} | {gender.capitalize() if gender else 'Unknown'}, {f'{age}y' if age else '?'}"
        if emotion:
            line1 += f" | {emotion.capitalize()}"

        if gazing:
            gaze_status = f"Gazing  {gaze_dur:.1f}s  (Total: {total_gaze:.1f}s)"
        else:
            gaze_status = "Not Gazing"

        line2 = (f"{gaze_status}  |  Yaw:{yaw:+.0f}  Pitch:{pitch:+.0f}"
                 if yaw is not None and pitch is not None else gaze_status)

        # BUG 2 FIX: use getTextSize to measure each line exactly,
        # then clamp tx so the widest line stays fully within the frame.
        (w1, _), _ = cv2.getTextSize(line1, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
        (w2, _), _ = cv2.getTextSize(line2, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
        max_w = max(w1, w2)
        tx = max(0, min(x1, PREVIEW_WIDTH - max_w - 5))

        cv2.putText(img, line1, (tx, y1-22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
        cv2.putText(img, line2, (tx, y1-6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, color, 1, cv2.LINE_AA)

        kpts = f.get("kpts")
        if kpts is not None and isinstance(kpts, np.ndarray) and kpts.shape==(5,2):
            for kp in kpts:
                cv2.circle(img, tuple(map(int,kp)), 2, (255,0,255), -1)
    return img

# ----------------------------------------------------------
# Load Models
# ----------------------------------------------------------
logger.info("Loading models...")
try:
    widerface_model  = dg.load_model(model_name=widerface_model_name,
        inference_host_address=inference_host_address, zoo_url=zoo_url, token=token, device_type=device_type)
    logger.info("Loaded WiderFace model")
    face_embed_model = dg.load_model(model_name=face_embed_model_name,
        inference_host_address=inference_host_address, zoo_url=zoo_url, token=token, device_type=device_type)
    logger.info("Loaded embedding model")
    age_model = dg.load_model(model_name=age_model_name,
        inference_host_address=inference_host_address, zoo_url=zoo_url, token=token, device_type=device_type)
    logger.info("Loaded age model")
    gender_model = dg.load_model(model_name=gender_model_name,
        inference_host_address=inference_host_address, zoo_url=zoo_url, token=token, device_type=device_type)
    logger.info("Loaded gender model")
    emotion_model = dg.load_model(model_name=emotion_model_name,
        inference_host_address=inference_host_address, zoo_url=zoo_url, token=token, device_type=device_type)
    logger.info("Loaded emotion model")
except Exception:
    logger.exception("Failed to load models")
    raise

# ----------------------------------------------------------
# Initialize
# ----------------------------------------------------------
camera      = ThreadedCamera(fps=CAMERA_FPS, width=PREVIEW_WIDTH, height=PREVIEW_HEIGHT)
tracker     = ViewerTracker(iou_thr=0.25, emb_thr=0.5, timeout=3.0)
gaze_mgr    = GazeSessionManager()
demo_worker = DemographicsWorker(age_model, gender_model, emotion_model, queue_size=DEMO_QUEUE_SIZE)

display_available = preview_camera
if preview_camera:
    try:
        cv2.namedWindow("test", cv2.WINDOW_NORMAL); cv2.destroyWindow("test")
        logger.info("Display available")
    except cv2.error:
        logger.warning("No display — preview disabled")
        display_available = False

fps_q = deque(maxlen=30)
last_fps_t = last_log_t = time.time()
frame_counter = total_faces_detected = total_gaze_events = 0

logger.info("=" * 60)
logger.info("GAZE TRACKING v3.5.6-PRODUCTION-STABLE STARTED")
logger.info(f"  Camera: {PREVIEW_WIDTH}x{PREVIEW_HEIGHT} @ {CAMERA_FPS}fps | Skip: {SKIP_FRAMES}")
logger.info(f"  Focal: {_FOCAL_LENGTH:.1f}px | PnP solver: {_SOLVEPNP_SOLVER_NAME} (standby)")
logger.info(f"  GAZE MODE: {GAZE_MODE}")
logger.info(f"    NOTE: Hailo WiderFace model does not expose facial landmarks on this")
logger.info(f"    SDK config. Gaze detection uses position-based fallback exclusively.")
logger.info(f"    PnP head-pose estimation is configured but will not activate.")
logger.info(f"  Screen bias: pitch={SCREEN_PITCH_BIAS_DEG:+.1f} yaw={SCREEN_YAW_BIAS_DEG:+.1f} (deg)")
logger.info(f"  Gaze cone: yaw +/-{GAZE_YAW_THRESHOLD}  pitch +/-{GAZE_PITCH_THRESHOLD} (compensated, deg)")
logger.info(f"  Fallback centre: X={FALLBACK_CENTER_RATIO_X} Y={FALLBACK_CENTER_RATIO_Y}")
logger.info(f"  Hysteresis: ON={GAZE_ON_FRAMES}fr OFF={GAZE_OFF_FRAMES}fr | Confirm={N_CONFIRM_FRAMES}fr")
logger.info(f"  Freeze hardening: BME688+noise background threads | embed timeout={EMBED_TIMEOUT_S}s | watchdog={WATCHDOG_TIMEOUT_S}s")
logger.info(f"  GAZE_DEBUG={GAZE_DEBUG}  (set True + look at screen centre to calibrate SCREEN_PITCH_BIAS_DEG)")
logger.info(f"  Log: {os.path.abspath(LOG_FILE)}")
logger.info("=" * 60)

# ----------------------------------------------------------
# Main Loop
# ----------------------------------------------------------
try:
    while True:
        loop_start   = time.time()
        frame        = camera.read()
        if frame is None:
            time.sleep(0.01); continue

        frame_counter += 1
        current_time   = time.time()
        faces          = []
        was_processed  = False

        dt = current_time - last_fps_t
        fps_q.append(1.0/dt if dt else 0)
        last_fps_t = current_time
        avg_fps    = sum(fps_q)/len(fps_q) if fps_q else 0

        # ---- INFERENCE ----
        if frame_counter % SKIP_FRAMES == 0:
            cpu_temp = get_cpu_temp()
            if cpu_temp >= THERMAL_CRITICAL_TEMP:
                logger.warning(f"CRITICAL {cpu_temp:.0f}°C — sleeping 1s"); time.sleep(1.0)
            elif cpu_temp >= THERMAL_THROTTLE_TEMP:
                logger.info(f"THROTTLE {cpu_temp:.0f}°C — sleeping 200ms"); time.sleep(0.2)

            if cpu_temp < THERMAL_CRITICAL_TEMP:
                try:
                    det = widerface_model.predict(frame)
                    was_processed = True
                    if hasattr(det, 'results'):
                        # Build ordered list (largest face first), tracking original index
                        det_items = []
                        for det_idx, d in enumerate(det.results):
                            try:
                                bbox = d.get("bbox") if isinstance(d,dict) else getattr(d,"bbox",None)
                                if bbox:
                                    x1,y1,x2,y2 = bbox
                                    det_items.append(((x2-x1)*(y2-y1), det_idx, d))
                            except Exception:
                                continue
                        det_items.sort(key=lambda x: x[0], reverse=True)

                        # FIX 1: Pre-extract batch-level kpts from `det` object.
                        # DeGirum may store all keypoints on the batch result,
                        # not inside each per-detection dict. Try every known
                        # field name and build a per-index lookup.
                        batch_kpts = {}   # {det_idx: raw_kpts_data}
                        if not _kpts_debug_logged:
                            # Enhanced KPTS_DEBUG: log batch result structure too
                            _log_result_structure(det, label="batch_result")

                        for fname in _KPTS_FIELD_NAMES:
                            batch_arr = getattr(det, fname, None)
                            if batch_arr is None and isinstance(det, dict):
                                batch_arr = det.get(fname)
                            if batch_arr is not None:
                                try:
                                    arr = np.array(batch_arr)
                                    # Shape (N,5,2), (N,5,3), (N,10), (N,15)
                                    if arr.ndim == 3 and arr.shape[1] == 5:
                                        for i in range(min(len(arr), len(det.results))):
                                            batch_kpts[i] = arr[i]
                                        if GAZE_DEBUG:
                                            logger.debug(f"Batch kpts found: field='{fname}' shape={arr.shape}")
                                        break
                                    elif arr.ndim == 2 and arr.shape[1] in (10, 15):
                                        for i in range(min(len(arr), len(det.results))):
                                            batch_kpts[i] = arr[i]
                                        break
                                except Exception:
                                    continue

                        for _, det_idx, d in det_items[:MAX_FACES_PER_FRAME]:
                            try:
                                bbox = d.get("bbox") if isinstance(d,dict) else getattr(d,"bbox",None)
                                x1,y1,x2,y2 = map(int, bbox)
                                x1,y1 = max(0,x1), max(0,y1)
                                x2,y2 = min(frame.shape[1],x2), min(frame.shape[0],y2)
                                if x2<=x1 or y2<=y1: continue
                                crop = frame[y1:y2, x1:x2]

                                # FIX 2: Enhanced KPTS_DEBUG — WARNING level, logs per-detection dict
                                if KPTS_DEBUG and not _kpts_debug_logged:
                                    _log_result_structure(d, label="per_detection_result")

                                # Try per-detection dict first (existing logic)
                                kr = None
                                if isinstance(d, dict):
                                    for fname in _KPTS_FIELD_NAMES:
                                        kr = d.get(fname)
                                        if kr is not None: break
                                    if kr is None:
                                        kr = d
                                else:
                                    for fname in _KPTS_FIELD_NAMES:
                                        kr = getattr(d, fname, None)
                                        if kr is not None: break
                                    if kr is None:
                                        kr = d

                                kpts = parse_keypoints(kr)

                                # FIX 1: If per-detection failed, try batch-level
                                if kpts is None and det_idx in batch_kpts:
                                    kpts = parse_keypoints(batch_kpts[det_idx])
                                    if kpts is not None and GAZE_DEBUG:
                                        logger.debug(f"kpts from batch level, det_idx={det_idx}")

                                # Scale normalized coords to pixels
                                kpts = auto_scale_kpts(kpts, bbox=bbox)

                                # v3.5.3 FIX: embed predict with hard timeout
                                emb_raw = {}
                                if crop.size > 0:
                                    _emb_result = {}
                                    def _do_embed(_crop=crop):
                                        try:
                                            er = face_embed_model.predict(_crop)
                                            if hasattr(er,'results') and er.results:
                                                _emb_result['data'] = er.results[0]
                                        except Exception:
                                            pass
                                    _et = threading.Thread(target=_do_embed, daemon=True)
                                    _et.start()
                                    _et.join(timeout=EMBED_TIMEOUT_S)
                                    if _et.is_alive():
                                        logger.warning("face_embed_model.predict() timed out")
                                    else:
                                        emb_raw = _emb_result.get('data', {})

                                _gaze_diag["frames_with_faces"] += 1
                                faces.append({
                                    "bbox": bbox, "kpts": kpts, "embedding": emb_raw,
                                    "crop": crop, "age_est": 0, "gender": "",
                                    "gender_score": 0.0, "emotion": "", "emotion_score": 0.0,
                                })
                            except Exception:
                                continue
                    total_faces_detected += len(faces)
                except Exception:
                    logger.exception("Detection failed")

        # ---- TRACKING + GAZE ----
        if was_processed and faces:
            bboxes = [f["bbox"] for f in faces]
            embs   = [extract_embedding(f.get("embedding",{})) for f in faces]
            assignments = tracker.update(bboxes, embs)

            if len(assignments) != len(faces):
                logger.warning(f"Track mismatch: {len(assignments)} vs {len(faces)}")
            else:
                for i, f in enumerate(faces):
                    try:
                        vid, is_new, is_confirmed = assignments[i]

                        # Initialise profile on first sight
                        if is_new or vid not in viewer_profiles:
                            viewer_profiles[vid] = {
                                'age': 0, 'gender': '', 'first_seen': current_time,
                                'emotions': {}, 'demo_submitted': False
                            }

                        # BUG 3 FIX: submit demographics when track is confirmed,
                        # not on first-new (which is always unconfirmed).
                        # Guard with demo_submitted flag to submit exactly once.
                        if is_confirmed and not viewer_profiles[vid].get('demo_submitted'):
                            crop = f.get("crop")
                            if crop is not None and crop.size > 0:
                                demo_worker.submit(vid, crop)
                                viewer_profiles[vid]['demo_submitted'] = True

                        f.pop("crop", None)
                        kpts = f.get("kpts")
                        bbox = f.get("bbox")
                        valid_kpts = (kpts is not None and
                                      isinstance(kpts, np.ndarray) and kpts.shape==(5,2))

                        gazing, yaw, pitch, confidence = is_gazing_at_screen(
                            kpts if valid_kpts else None, bbox, viewer_id=vid)

                        total_g, cont_g, is_new_gaze = gaze_mgr.update(
                            vid, gazing, current_time, confidence=confidence)

                        sm_state       = gaze_mgr.sessions.get(vid,{}).get('state','NOT_GAZING')
                        gaze_confirmed = (sm_state == 'GAZING')

                        f["is_gazing"]      = gaze_confirmed
                        f["gaze_confirmed"] = gaze_confirmed
                        f["gaze_duration"]  = cont_g
                        # BUG 2 FIX: include current open bout in total display
                        f["total_gaze"]     = total_g + cont_g
                        f["yaw"]            = yaw
                        f["pitch"]          = pitch
                        f["viewer_id"]      = vid

                        prof = viewer_profiles[vid]
                        if prof['age']:    f["age_est"] = prof['age']
                        if prof['gender']: f["gender"]  = prof['gender']
                        emo = f.get("emotion","")
                        if not emo and prof.get('emotions'):
                            emo = max(prof['emotions'].items(), key=lambda x:x[1])[0]
                            f["emotion"] = emo

                        if is_new_gaze:
                            total_gaze_events += 1
                            log_gaze_event({
                                'timestamp':  datetime.utcnow().isoformat()+"Z",
                                'event':      'gaze_start',
                                'viewer_id':  vid,
                                'demographics':{'age':prof['age'],'gender':prof['gender'],'emotion':emo},
                                'head_pose':  {'yaw':  round(yaw,2)   if yaw   is not None else None,
                                               'pitch':round(pitch,2) if pitch is not None else None},
                                'gaze_confidence': confidence,
                            })
                            logger.info(f"GAZE_START: {vid} ({prof['gender']} {prof['age']}y) conf={confidence:.2f}")

                    except Exception:
                        logger.debug(f"Face {i} error", exc_info=True)
                        continue

        # ---- HEARTBEAT ----
        if current_time - last_log_t >= LOG_INTERVAL:
            active = []
            for vid, s in gaze_mgr.sessions.items():
                if s['state']=='GAZING' and s['start']:
                    ct   = current_time - s['start']
                    p    = viewer_profiles.get(vid,{})
                    emos = p.get('emotions',{})
                    if ct >= MIN_GAZE_DURATION:
                        active.append({
                            'viewer_id':       vid,
                            'continuous_gaze': round(ct,1),
                            'total_gaze':      round(s['total']+ct,1),
                            'demographics':    {'age':p.get('age',0),'gender':p.get('gender',''),
                                                'emotion':max(emos.items(),key=lambda x:x[1])[0] if emos else ''}
                        })

            env      = read_bme688_data()
            noise_db = read_noise_level_db()
            if noise_db is not None:
                env["noise_db"] = noise_db
            gaze_diag = get_gaze_diagnostics()

            heartbeat = {
                'timestamp':            datetime.utcnow().isoformat()+"Z",
                'event':                'heartbeat',
                'active_gazers':        len(active),
                'tracked_viewers':      len(gaze_mgr.sessions),
                'total_faces_detected': total_faces_detected,
                'total_gaze_events':    total_gaze_events,
                'fps':                  round(avg_fps,1),
                'cpu_temp':             round(get_cpu_temp(),1),
                'environment':          env,
                'gaze_diagnostics':     gaze_diag,
            }
            if active: heartbeat['gazers'] = active
            log_gaze_event(heartbeat)
            logger.info(f"HEARTBEAT: gazers={len(active)} tracked={len(gaze_mgr.sessions)} "
                        f"faces_total={total_faces_detected} gaze_events={total_gaze_events} "
                        f"FPS={avg_fps:.1f} CPU={get_cpu_temp():.0f}°C "
                        f"kpts_valid={gaze_diag['kpts_valid_pct']}% "
                        f"pnp_ok={gaze_diag['solvepnp_success_pct']}% "
                        f"fallback={gaze_diag['fallback_pct']}%")
            cleanup_pose_history(set(gaze_mgr.sessions.keys()))
            last_log_t = current_time

        # ---- STALE CLEANUP ----
        for ss in gaze_mgr.cleanup_stale():
            vid = ss['viewer_id']
            p   = viewer_profiles.get(vid,{})
            log_gaze_event({
                'timestamp':     datetime.utcnow().isoformat()+"Z",
                'event':         'session_end',
                'viewer_id':     vid,
                'session_stats': ss,
                'demographics':  {'age':p.get('age',0),'gender':p.get('gender',''),'emotions':p.get('emotions',{})},
                'environment':   read_bme688_data(),
            })
            logger.info(f"SESSION END: {vid} gaze={ss['total_gaze_time']:.1f}s "
                        f"dur={ss['session_duration']:.1f}s engage={ss['engagement_rate']:.1%}")
            viewer_profiles.pop(vid, None)

        # ---- PREVIEW ----
        if display_available:
            disp = frame.copy()
            if faces: disp = draw_overlay(disp, faces)
            n_gaze    = sum(1 for f in faces if f.get("is_gazing"))
            diag      = get_gaze_diagnostics()
            env       = read_bme688_data()
            noise_db  = read_noise_level_db()
            hud_line1 = (f"FPS: {avg_fps:.1f} | Faces: {len(faces)} | Gaze: {n_gaze} | "
                         f"Hum: {env.get('humidity',0.0):.0f}% | "
                         f"Temp: {env.get('temp_c',0.0):.1f}C | "
                         f"dB: {f'{noise_db:.1f}' if noise_db is not None else 'N/A'} ")
            hud_line2 = (f"Tracked:{len(gaze_mgr.sessions)} | "
                         f"Mode:{GAZE_MODE} | "
                         f"Events:{total_gaze_events} | "
                         f"CPU:{get_cpu_temp():.0f}C")
            cv2.putText(disp, hud_line1, (10,28), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,0), 1, cv2.LINE_AA)
            cv2.putText(disp, hud_line2, (10,52), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180,220,180), 1, cv2.LINE_AA)
            cv2.imshow("Audience Gaze Analysis", disp)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                logger.info("User exit"); break

        elapsed = time.time() - loop_start
        if elapsed > WATCHDOG_TIMEOUT_S:
            logger.warning(f"WATCHDOG: main loop took {elapsed:.2f}s — possible slow I/O or inference hang")
        if elapsed < MIN_LOOP_TIME:
            time.sleep(MIN_LOOP_TIME - elapsed)

except KeyboardInterrupt:
    logger.info("Interrupted")
finally:
    logger.info("Shutting down...")
    logger.info(f"FINAL GAZE DIAGNOSTICS: {json.dumps(get_gaze_diagnostics())}")
    for vid in list(gaze_mgr.sessions.keys()):
        st = gaze_mgr.end_session(vid)
        if st:
            p = viewer_profiles.get(vid,{})
            log_gaze_event({
                'timestamp':     datetime.utcnow().isoformat()+"Z",
                'event':         'shutdown_session_end',
                'viewer_id':     vid,
                'session_stats': st,
                'demographics':  {'age':p.get('age',0),'gender':p.get('gender',''),'emotions':p.get('emotions',{})},
            })
            logger.info(f"Final: {vid} gaze={st['total_gaze_time']:.1f}s "
                        f"dur={st['session_duration']:.1f}s engage={st['engagement_rate']:.1%}")
    demo_worker.stop()
    camera.stop()
    if display_available:
        cv2.destroyAllWindows()
    logger.info(f"Clean shutdown. faces={total_faces_detected} gaze_events={total_gaze_events}")

2026-03-12 19:45:01,574 [INFO] === Log initialized: /home/william/ai_digital_signage/microservices/audience-analysis-service/logs/audience_analysis_live.log ===
2026-03-12 19:45:01,657 [INFO] BME688 initialized (secondary)
2026-03-12 19:45:01,675 [INFO] Loading models...
2026-03-12 19:45:01,823 [INFO] Loaded WiderFace model


Exception in thread bme688:
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1038, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.11/threading.py", line 975, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_6650/1459231375.py", line 528, in _bme688_background_loop
NameError: name 'BME_READ_INTERVAL' is not defined. Did you mean: 'NOISE_READ_INTERVAL'?


2026-03-12 19:45:01,878 [INFO] Loaded embedding model
2026-03-12 19:45:01,942 [INFO] Loaded age model
2026-03-12 19:45:01,994 [INFO] Loaded gender model
2026-03-12 19:45:02,054 [INFO] Loaded emotion model


[0:30:11.847345078] [6650]  INFO Camera camera_manager.cpp:330 libcamera v0.5.2+99-bfd68f78
[0:30:11.856850887] [6755]  INFO RPI pisp.cpp:720 libpisp version v1.2.1 981977ff21f3 29-04-2025 (14:13:50)
[0:30:11.860265791] [6755]  INFO IPAProxy ipa_proxy.cpp:180 Using tuning file /usr/share/libcamera/ipa/rpi/pisp/imx708.json
[0:30:11.868586082] [6755]  INFO Camera camera_manager.cpp:220 Adding camera '/base/axi/pcie@1000120000/rp1/i2c@88000/imx708@1a' for pipeline handler rpi/pisp
[0:30:11.868616009] [6755]  INFO RPI pisp.cpp:1179 Registered camera /base/axi/pcie@1000120000/rp1/i2c@88000/imx708@1a to CFE device /dev/media0 and ISP device /dev/media1 using PiSP variant BCM2712_D0
[0:30:11.873767253] [6650]  INFO Camera camera.cpp:1215 configuring streams: (0) 640x480-RGB888/sRGB (1) 1536x864-BGGR_PISP_COMP1/RAW
[0:30:11.873895032] [6755]  INFO RPI pisp.cpp:1483 Sensor: /base/axi/pcie@1000120000/rp1/i2c@88000/imx708@1a - Selected sensor format: 1536x864-SBGGR10_1X10/RAW - Selected CFE forma

2026-03-12 19:45:03,114 [INFO] Camera 640x480 @ 10fps
2026-03-12 19:45:03,120 [INFO] Demographics worker started
2026-03-12 19:45:03,263 [INFO] Display available
2026-03-12 19:45:03,269 [INFO] ============================================================
2026-03-12 19:45:03,276 [INFO] GAZE TRACKING v3.5.6-PRODUCTION-STABLE STARTED
2026-03-12 19:45:03,280 [INFO]   Camera: 640x480 @ 10fps | Skip: 3
2026-03-12 19:45:03,285 [INFO]   Focal: 482.5px | PnP solver: SQPNP (standby)
2026-03-12 19:45:03,289 [INFO]   GAZE MODE: POS-FALLBACK
2026-03-12 19:45:03,295 [INFO]     NOTE: Hailo WiderFace model does not expose facial landmarks on this
2026-03-12 19:45:03,301 [INFO]     SDK config. Gaze detection uses position-based fallback exclusively.
2026-03-12 19:45:03,307 [INFO]     PnP head-pose estimation is configured but will not activate.
2026-03-12 19:45:03,314 [INFO]   Screen bias: pitch=+15.0 yaw=+0.0 (deg)
2026-03-12 19:45:03,318 [INFO]   Gaze cone: yaw +/-30  pitch +/-20 (compensated, deg)
20